# Debt-to-Income Ratio Impact Analysis

### Retail Loan Mortgage Approval — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), and basic understanding of retail lending and mortgage approval terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Describe the information-extraction task and its relevance to retail lending and mortgage approval.
- Implement a rule-based extraction pipeline and evaluate accuracy on synthetic examples.
- Assess limitations of rule-based extraction and when ML-based approaches would be more appropriate.

**Estimated time:** 35–45 minutes

**Collection context:** Mortgage risk, automated underwriting, appraisal valuation, and loan approval processes in retail banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** DTI is the ultimate measure of affordability. It asks: "After paying all your monthly debts, do you have enough left for food and life?" If DTI is too high, the borrower is 'house poor' and extremely likely to default if they have even a small financial emergency.

**Input data used:** Monthly Gross Income, Existing Monthly Debts, Proposed Mortgage Payment.

**What we calculate:** Total DTI and the 'Marginal Risk' of adding $100 more in debt.

**Math method used:** Logistic Probability Mapping and Visualization.

**Learning goal:** Understand how 'Affordability Guardrails' protect both the bank and the borrower.

**Primary stakeholders:** mortgage officers, underwriters, credit risk managers, and compliance teams.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic DTI Data

We simulate a range of incomes and debt levels to see where the 'danger zone' starts.

In [ ]:
n = 2000
monthly_income = RNG.uniform(2000, 15000, n)
existing_debt = RNG.uniform(0, 3000, n)
mortgage_payment = RNG.uniform(1000, 5000, n)

total_monthly_debt = existing_debt + mortgage_payment
dti = total_monthly_debt / monthly_income

# Risk Logic:
# Risk is low until DTI hits 0.36 (36% is a common limit).
# Risk becomes extreme after 0.50.
risk_prob = 1 / (1 + np.exp(-15 * (dti - 0.43))) + RNG.normal(0, 0.05, n)
risk_prob = np.clip(risk_prob, 0, 1)

df = pd.DataFrame({
    'monthly_income': monthly_income,
    'total_debt': total_monthly_debt,
    'dti': dti,
    'default_risk': risk_prob
})

## Step 4 — Exploratory Data Analysis

Analyze the 'Tipping Point'

Let's look at how the average risk changes as DTI increases.

In [ ]:
df['dti_bucket'] = pd.cut(df['dti'], bins=np.arange(0, 1.1, 0.1))
summary = df.groupby('dti_bucket')['default_risk'].mean()

plt.figure(figsize=(10, 6))
summary.plot(kind='bar', color='orange')
plt.axvline(x=3.5, color='#E76F51', linestyle='--', label='Policy Limit (43%)')
plt.title('Average Default Risk by DTI Ratio')
plt.ylabel('Probability of Default')
plt.xlabel('Debt-to-Income (DTI) Ratio')
plt.legend()
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Visualisation titled "Average Default Risk by DTI Ratio" with Debt-to-Income (DTI) Ratio on the x-axis and Probability of Default on the y-axis. The chart highlights key patterns that inform the modelling approach.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

For this workflow, the data prepared in Step 3 is used directly as input features.

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: manual extraction (no automation) ---
print("Baseline: without automation, each document must be read and keyed manually.")
print("Any automated approach that achieves >80% field accuracy saves significant time.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
# Create a grid for the heatmap
incomes = np.linspace(2000, 15000, 20)
debts = np.linspace(0, 8000, 20)
Z = np.zeros((len(debts), len(incomes)))

for i, d in enumerate(debts):
    for j, inc in enumerate(incomes):
        ratio = d / inc
        prob = 1 / (1 + np.exp(-15 * (ratio - 0.43)))
        Z[i, j] = prob

plt.figure(figsize=(12, 8))
sns.heatmap(Z, xticklabels=incomes.astype(int), yticklabels=debts.astype(int), cmap='RdYlGn_r')
plt.title('Lending Risk Heatmap (Red = Dangerous DTI)')
plt.xlabel('Monthly Gross Income ($)')
plt.ylabel('Total Monthly Debt ($)')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Heatmap titled "Lending Risk Heatmap (Red = Dangerous DTI)" with Monthly Gross Income ($) on the x-axis and Total Monthly Debt ($) on the y-axis. The heatmap reveals correlation strength and direction between variables.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

1. **The 43% Rule:** Many mortgage programs (like FHA) have a hard limit at 43% DTI. Our analysis shows why—the risk of default starts to rise exponentially after this point.
2. **Income Sensitivity:** Notice that low-income borrowers have much less 'room for error'. A $100 increase in debt for someone earning $3k/mo has a much bigger impact on DTI than for someone earning $15k/mo.
3. **Conclusion:** DTI is a simple but powerful metric. By enforcing DTI limits, banks ensure that borrowers have a 'margin of safety' to handle unexpected life events (medical bills, car repairs) without missing their mortgage payment.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: process a new document ---
print("Operational demonstration — field extraction:\n")
new_doc = "Invoice #2055. Vendor: Wayne Enterprises. Date: 2024-06-15. Total: $8750.00"
result = extract_info(new_doc)
print(f"  Raw: {new_doc}")
print(f"  Vendor: {result[0]}  Date: {result[1]}  Amount: ${result[2]:,.2f}")
print("\nExtracted fields would auto-populate the accounts-payable system.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end retail lending and mortgage approval workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Mortgage models must comply with fair lending laws and avoid discriminatory approval patterns.
- Automated underwriting can disadvantage applicants from historically underserved communities.
- Transparency in loan decisions is a regulatory requirement, not an optional feature.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in retail lending and mortgage approval?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.